In [ ]:
%pip install seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

# Create synthetic dataset if files don't exist
if os.path.exists("application_record.csv") and os.path.exists("credit_record.csv"):
    app = pd.read_csv("application_record.csv")
    credit = pd.read_csv("credit_record.csv")
else:
    print("Creating synthetic dataset for demonstration...")
    
    # Create application records
    np.random.seed(42)
    n_samples = 1000
    
    app = pd.DataFrame({
        'ID': range(1, n_samples + 1),
        'CODE_GENDER': np.random.choice(['M', 'F'], n_samples),
        'FLAG_OWN_CAR': np.random.choice(['Y', 'N'], n_samples),
        'FLAG_OWN_REALTY': np.random.choice(['Y', 'N'], n_samples),
        'CNT_CHILDREN': np.random.randint(0, 5, n_samples),
        'AMT_INCOME_TOTAL': np.random.randint(25000, 500000, n_samples),
        'NAME_INCOME_TYPE': np.random.choice(['Working', 'Commercial', 'Pensioner'], n_samples),
        'NAME_EDUCATION_TYPE': np.random.choice(['Secondary', 'Higher', 'Incomplete higher'], n_samples),
        'NAME_FAMILY_STATUS': np.random.choice(['Single', 'Married', 'Widowed', 'Divorced'], n_samples),
        'NAME_HOUSING_TYPE': np.random.choice(['House', 'Apartment', 'Co-op'], n_samples),
        'DAYS_BIRTH': np.random.randint(-25000, -7000, n_samples),
        'DAYS_EMPLOYED': np.random.randint(-10000, 0, n_samples),
        'OCCUPATION_TYPE': np.random.choice(['Accountants', 'Managers', 'Engineers', 'Sales', None], n_samples)
    })
    
    # Create credit records
    credit = pd.DataFrame({
        'ID': np.random.choice(range(1, n_samples + 1), n_samples * 5),
        'MONTHS_BALANCE': np.random.randint(-12, 0, n_samples * 5),
        'STATUS': np.random.choice(['0', '1', '2', 'C', 'X', 'Default'], n_samples * 5)
    })

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

# =====================================================
# Visualization
# =====================================================

print(app.head())
print(app["OCCUPATION_TYPE"].value_counts())

sns.set(rc={'figure.figsize':(18,6)})
sns.countplot(x="OCCUPATION_TYPE",
              data=app,
              palette="Set2")

plt.xticks(rotation=90)
plt.show()

plt.figure(figsize=(10,8))
sns.heatmap(app.select_dtypes(include=np.number).corr(),
            annot=True,
            cmap="Blues")
plt.show()

print(app.describe())

# =====================================================
# Remove Duplicates
# =====================================================

app.drop_duplicates(inplace=True)

# =====================================================
# Missing Values
# =====================================================

app["OCCUPATION_TYPE"] = app["OCCUPATION_TYPE"].fillna("Unknown")

# =====================================================
# Convert Status to Binary
# =====================================================

def to_binary(x):
    if x in ["0","X","C"]:
        return 1
    else:
        return 0

credit["STATUS_BIN"] = credit["STATUS"].apply(to_binary)

# =====================================================
# Aggregate Credit Data
# =====================================================

credit = credit.groupby("ID")["STATUS_BIN"].max().reset_index()

# =====================================================
# Merge
# =====================================================

final_df = app.merge(credit, on="ID", how="inner")

print(final_df.head())

# =====================================================
# Label Encoding
# =====================================================

columns = [
"CODE_GENDER",
"FLAG_OWN_CAR",
"FLAG_OWN_REALTY",
"NAME_INCOME_TYPE",
"NAME_EDUCATION_TYPE",
"NAME_FAMILY_STATUS",
"NAME_HOUSING_TYPE",
"OCCUPATION_TYPE"
]

encoder = LabelEncoder()

for col in columns:
    final_df[col] = encoder.fit_transform(final_df[col])

# =====================================================
# Features and Target
# =====================================================

X = final_df.drop(["ID","STATUS_BIN"],axis=1)
y = final_df["STATUS_BIN"]

# =====================================================
# Train Test Split
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# =====================================================
# Logistic Regression
# =====================================================

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print("\nLOGISTIC REGRESSION")
print(classification_report(y_test, lr_pred))
lr_acc = accuracy_score(y_test, lr_pred)

# =====================================================
# Decision Tree
# =====================================================

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

print("\nDECISION TREE")
print(classification_report(y_test, dt_pred))
dt_acc = accuracy_score(y_test, dt_pred)

# =====================================================
# Random Forest
# =====================================================

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("\nRANDOM FOREST")
print(classification_report(y_test, rf_pred))
rf_acc = accuracy_score(y_test, rf_pred)

# =====================================================
# Accuracy Comparison
# =====================================================

print("\nAccuracy Scores")
print("-----------------------")
print("Logistic Regression :", lr_acc)
print("Decision Tree       :", dt_acc)
print("Random Forest       :", rf_acc)

# =====================================================
# Save Best Model
# =====================================================

models = {
    "Logistic Regression": (lr, lr_acc),
    "Decision Tree": (dt, dt_acc),
    "Random Forest": (rf, rf_acc)
}

best_name, (best_model, best_acc) = max(
    models.items(),
    key=lambda x: x[1][1]
)

print("\nBest Model :", best_name)
print("Accuracy :", best_acc)

pickle.dump(best_model, open("model.pkl", "wb"))

print("\nmodel.pkl saved successfully.")

# =====================================================
# Test Saved Model
# =====================================================

loaded_model = pickle.load(open("model.pkl", "rb"))

print("\nSample Prediction")
print(loaded_model.predict(X_test.iloc[:5]))



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


RuntimeError: Failed to load dataset. Please ensure 'application_record.csv' and 'credit_record.csv' are in the current directory, or configure Kaggle API (provide kaggle.json).